# Triton AWS DLC — Batch Transform

Usa a imagem `sagemaker-tritonserver` mantida pela AWS.  
- Sem Dockerfile próprio, sem push para ECR
- `/ping` e `/invocations` já mapeados pelo middleware da imagem
- `BatchStrategy: SingleRecord` — cada linha do JSONL vira um request independente

## Local (teste direto via Triton HTTP)

In [ ]:
import requests

In [ ]:
# Container local iniciado com a imagem AWS DLC:
#   docker run --gpus all -p 8080:8080 \
#     -e SAGEMAKER_TRITON_DEFAULT_MODEL_NAME=pipeline \
#     -v "$HOME/.aws:/root/.aws:ro" \
#     -v "$(pwd)/app/triton_aws/model_repository:/opt/ml/model" \
#     007439368137.dkr.ecr.us-east-1.amazonaws.com/sagemaker-tritonserver:26.03-py3 serve
#
# A imagem AWS usa /invocations (não /v2/models/pipeline/infer)

payload = {
    "inputs": [
        {"name": "record_id",     "shape": [1, 1], "datatype": "BYTES", "data": ["cardboard1"]},
        {"name": "image_s3_path", "shape": [1, 1], "datatype": "BYTES",
         "data": ["s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/cardboard1.jpg"]}
    ]
}

resp = requests.post("http://localhost:8080/invocations", json=payload)
resp.raise_for_status()
resp.json()

## AWS — Batch Transform

In [36]:
import boto3
import json
import time
import random
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse

In [37]:
session    = boto3.session.Session(region_name="us-east-1")
region     = session.region_name or "us-east-1"
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Região:     {region}")
print(f"Account ID: {account_id}")

Região:     us-east-1
Account ID: 891377318910


In [38]:

# Imagem própria no ECR: AWS DLC 24.09-py3 + boto3 + Pillow.
# Evita instalar dependências em runtime; compatível com kernel driver 470.x (ml.g4dn.xlarge).
ECR_IMAGE_URI = f"{account_id}.dkr.ecr.{region}.amazonaws.com/sm-batch-transform-triton-aws:latest"

SAGEMAKER_ROLE_ARN = f"arn:aws:iam::{account_id}:role/SageMakerExecutionRole-DSA-P3"

S3_BUCKET      = "mlops-us-east-1-891377318910"
S3_MODEL_KEY   = "sagemaker-batch-transform/refinamento/model/model_triton_aws.tar.gz"
MODEL_DATA_URL = f"s3://{S3_BUCKET}/{S3_MODEL_KEY}"

INPUT_S3  = f"s3://{S3_BUCKET}/sagemaker-batch-transform/refinamento/input/"
OUTPUT_S3 = f"s3://{S3_BUCKET}/sagemaker-batch-transform/refinamento/output/output-triton-aws/"

print(f"Imagem ECR:     {ECR_IMAGE_URI}")
print(f"Role ARN:       {SAGEMAKER_ROLE_ARN}")
print(f"Model data URL: {MODEL_DATA_URL}")
print(f"Input S3:       {INPUT_S3}")
print(f"Output S3:      {OUTPUT_S3}")


Imagem ECR:     891377318910.dkr.ecr.us-east-1.amazonaws.com/sm-batch-transform-triton-aws:latest
Role ARN:       arn:aws:iam::891377318910:role/SageMakerExecutionRole-DSA-P3
Model data URL: s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/model/model_triton_aws.tar.gz
Input S3:       s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/
Output S3:      s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/output/output-triton-aws/


### Criar Input Metadata File

In [39]:
_parsed = urlparse(INPUT_S3)
_bucket = _parsed.netloc
_prefix = _parsed.path.lstrip("/")

_pag = boto3.client("s3").get_paginator("list_objects_v2")
all_image_keys = [
    obj["Key"]
    for page in _pag.paginate(Bucket=_bucket, Prefix=_prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"Imagens disponíveis: {len(all_image_keys)}")

Imagens disponíveis: 45


In [40]:
# Cada linha é um request KServe v2 completo (formato que o /invocations recebe)
_sample = random.sample(all_image_keys, len(all_image_keys))

_lines = [
    json.dumps({
        "inputs": [
            {"name": "record_id",     "shape": [1, 1], "datatype": "BYTES", "data": [f"r{i:04d}"]},
            {"name": "image_s3_path", "shape": [1, 1], "datatype": "BYTES",
             "data": [f"s3://{_bucket}/{key}"]}
        ]
    })
    for i, key in enumerate(_sample)
]

_metadata_key = "sagemaker-batch-transform/refinamento/metadata/metadata_triton_aws.jsonl"
# boto3.client("s3").put_object(
#     Bucket=_bucket,
#     Key=_metadata_key,
#     Body="\n".join(_lines).encode("utf-8"),
#     ContentType="application/jsonlines",
# )

metadata_s3_uri = f"s3://{_bucket}/{_metadata_key}"
print(f"Metadata S3 URI: {metadata_s3_uri}")
print(f"\nExemplo da 1ª linha:\n{_lines[0]}")

Metadata S3 URI: s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/metadata/metadata_triton_aws.jsonl

Exemplo da 1ª linha:
{"inputs": [{"name": "record_id", "shape": [1, 1], "datatype": "BYTES", "data": ["r0000"]}, {"name": "image_s3_path", "shape": [1, 1], "datatype": "BYTES", "data": ["s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/metal4.jpg"]}]}


### Criar SageMaker Model

In [41]:
sm_client = boto3.client("sagemaker", region_name=region)

In [42]:
timestamp  = datetime.now().strftime("%Y%m%d%H%M%S")
model_name = f"resnet18-triton-aws-{timestamp}"
print(f"Nome do model: {model_name}")

Nome do model: resnet18-triton-aws-20260621192739


In [43]:
response_model = sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": ECR_IMAGE_URI,
        "ModelDataUrl": MODEL_DATA_URL,
        "Environment": {
            # Obrigatório para ensemble: diz ao middleware qual modelo expor em /invocations
            "SAGEMAKER_TRITON_DEFAULT_MODEL_NAME": "pipeline",
            "ALLOWED_S3_BUCKETS": S3_BUCKET,
            # Shared memory para os Python backends (preprocess/postprocess)
            "SAGEMAKER_TRITON_SHM_DEFAULT_BYTE_SIZE": "67108864",
        },
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN,
)

model_arn = response_model["ModelArn"]
print(f"Model ARN: {model_arn}")

Model ARN: arn:aws:sagemaker:us-east-1:891377318910:model/resnet18-triton-aws-20260621192739


### Criar Batch Transform Job

In [ ]:
job_name       = f"batch-triton-aws-{timestamp}"
instance_type  = "ml.g4dn.xlarge"
instance_count = 1

response_job = sm_client.create_transform_job(
    TransformJobName=job_name,
    ModelName=model_name,
    MaxConcurrentTransforms=8,  # aqui vai definir o batch do Triton (dynamic_batching {preferred_batch_size: [2, 4, 8])
    # SingleRecord: cada linha do JSONL → um POST /invocations → Triton processa 1 imagem
    # O middleware da imagem AWS mapeia /invocations → /v2/models/pipeline/infer
    BatchStrategy="SingleRecord",
    MaxPayloadInMB=6, # tamanho tem que ser suficiente para suportar a quantidade maxima de concorrencia
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": metadata_s3_uri,
            }
        },
        "ContentType": "application/json",
        "SplitType": "Line",
    },
    TransformOutput={
        "S3OutputPath": OUTPUT_S3,
        "AssembleWith": "Line",
    },
    TransformResources={
        "InstanceType": instance_type,
        "InstanceCount": instance_count,
    },
)

job_arn = response_job["TransformJobArn"]
print(f"Nome do job: {job_name}")
print(f"Job ARN:     {job_arn}")

Nome do job: batch-triton-aws-20260621192739
Job ARN:     arn:aws:sagemaker:us-east-1:891377318910:transform-job/batch-triton-aws-20260621192739


### Monitorar Status do Job

In [ ]:
while True:
    desc = sm_client.describe_transform_job(TransformJobName=job_name)
    status = desc["TransformJobStatus"]
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {status}")
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(30)

if desc.get("FailureReason"):
    print(f"Falha: {desc['FailureReason']}")
else:
    print(f"Início: {desc.get('TransformStartTime')}")
    print(f"Fim:    {desc.get('TransformEndTime')}")

### Ler Resultados do S3

In [ ]:
_parsed_out = urlparse(OUTPUT_S3)
_out_bucket = _parsed_out.netloc
_out_prefix = _parsed_out.path.lstrip("/")

_s3  = boto3.client("s3")
_pag = _s3.get_paginator("list_objects_v2")

output_keys = [
    obj["Key"]
    for page in _pag.paginate(Bucket=_out_bucket, Prefix=_out_prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].endswith((".jsonl", ".out"))
]

print(f"Arquivos de saída: {len(output_keys)}")
for k in output_keys:
    print(f"  s3://{_out_bucket}/{k}")

In [ ]:
records = []
for key in output_keys:
    body = _s3.get_object(Bucket=_out_bucket, Key=key)["Body"].read().decode()
    for line in body.strip().splitlines():
        if not line:
            continue
        resp_json = json.loads(line)
        out = {o["name"]: o["data"][0] for o in resp_json.get("outputs", [])}
        records.append(out)

df = pd.DataFrame(records)[["record_id", "class_name", "score"]]
df["score"] = df["score"].astype(float)
df = df.sort_values("record_id").reset_index(drop=True)

print(f"Total de registros: {len(df)}")

In [ ]:
df